# Embedding Lookup and Linear-Layer Shapes

## Objective
Verify embedding lookup, one-hot equivalence, linear transformations, broadcasting, and tensor reductions.

## Configuration
- Batch size: 2
- Tokens per sequence: 3
- Vocabulary size: 7
- Embedding dimension: 4
- Output features per token: 2

## Shape Flow

| Tensor / operation | Shape |
|---|---|
| Token IDs | `(2, 3)` |
| Embedding table | `(7, 4)` |
| One-hot token representation | `(2, 3, 7)` |
| Embedding lookup | `(2, 3, 4)` |
| Linear weights | `(4, 2)` |
| Bias | `(2,)` |
| Linear output | `(2, 3, 2)` |
| Mean over tokens | `(2, 2)` |
| Flatten tokens and features | `(2, 6)` |

## Validation
Embedding lookup and one-hot matrix multiplication produced identical results. The linear outputs, token means, and flattened values matched the supplied test case.

The same bias vector is broadcast across every batch and token position.

## Scope
This exercise checks forward computation and shape reasoning; it does not train an embedding model.

In [1]:
import torch

In [2]:
B, T, V, E, O = 2, 3, 7, 4, 2

token_ids = torch.tensor([
    [0, 2, 4],
    [1, 3, 6],
])

embedding_table = torch.tensor([
    [0., 1., 2., 3.],
    [1., 2., 3., 4.],
    [2., 3., 4., 5.],
    [3., 4., 5., 6.],
    [4., 5., 6., 7.],
    [5., 6., 7., 8.],
    [6., 7., 8., 9.],
])

weights = torch.tensor([
    [1., 0.],
    [0., 1.],
    [1., 0.],
    [0., 1.],
])

bias = torch.tensor([0.5, -0.5])

In [6]:
token_ids.shape, embedding_table.shape, weights.shape, bias.shape

(torch.Size([2, 3]), torch.Size([7, 4]), torch.Size([4, 2]), torch.Size([2]))

In [15]:
embedding_lookup = embedding_table[token_ids]

one_hot = torch.nn.functional.one_hot(token_ids, num_classes=V).float()
one_hot_emb = one_hot @ embedding_table

is_equivalent = torch.allclose(embedding_lookup, one_hot_emb)

z = embedding_lookup @ weights + bias

z_mean = z.mean(dim=1)
z_flat = z.flatten(start_dim=1)

print(f"""
    one hot embedding shape : {one_hot_emb.shape},
    embedding lookup shape : {embedding_lookup.shape},
    one hot and embedding lookup equivalance : {is_equivalent},
    output shape : {z.shape},
    token mean : {z_mean}, shape : {z_mean.shape}
    flatten : {z_flat}, shape : {z_flat.shape}
    """)


    one hot embedding shape : torch.Size([2, 3, 4]),
    embedding lookup shape : torch.Size([2, 3, 4]),
    one hot and embedding lookup equivalance : True,
    output shape : torch.Size([2, 3, 2]),
    token mean : tensor([[ 6.5000,  7.5000],
        [ 9.1667, 10.1667]]), shape : torch.Size([2, 2])
    flatten : tensor([[ 2.5000,  3.5000,  6.5000,  7.5000, 10.5000, 11.5000],
        [ 4.5000,  5.5000,  8.5000,  9.5000, 14.5000, 15.5000]]), shape : torch.Size([2, 6])
    
